In [ ]:
"""
SQUID Measurement Analysis — Batch Averaging & Background Correction
=====================================================================
Loads multiple raw SQUID .dat files for a single particle batch,
optionally subtracts the diamagnetic substrate background, averages
repeated measurements, and exports plots and a statistical report.

Background correction strategy (when enabled):
  A linear slope is fitted independently to the positive tail (top 5% of
  field range) and negative tail (bottom 5%) where FePt is fully saturated.
  The two slopes are averaged and subtracted, isolating the ferromagnetic
  FePt signal from the non-magnetic SiO2 substrate / holder.

Panel layout:
  1. Raw emu          — individual curves + mean ± 1 SD (no correction)
  2. Normalised M/Msat — slope-corrected individual curves + mean ± 1 SD
  3. Per-particle moment (emu / N_spheres, HCP packing) + mean ± 1 SD

Inputs  : Raw SQUID .dat files (Quantum Design MPMS3 format, [Data] header)
Outputs : PNG + SVG plot, .txt statistical report, averaged data CSV
          — all saved to a timestamped batch folder.

# ─────────────────────────────────────────────────────────────────────────────
# Copyright (C) 2026 Max-Planck-Gesellschaft zur Förderung der Wissenschaften e.V.
#               2026 University of Stuttgart
# Authors: Natalia Gonzalez-Vazquez, Andrew K. Schulz
# SPDX-License-Identifier: GPL-3.0-or-later
#
# This file is part of FunMaP.
# FunMaP is free software: you can redistribute it and/or modify it under
# the terms of the GNU General Public License as published by the Free
# Software Foundation, either version 3 of the License, or (at your option)
# any later version.
#
# FunMaP is distributed in the hope that it will be useful, but WITHOUT ANY
# WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS
# FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License along with
# FunMaP. If not, see <https://www.gnu.org/licenses/>.
# ─────────────────────────────────────────────────────────────────────────────
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- GUI ---
import tkinter as tk
from tkinter import filedialog, simpledialog, messagebox


# ===========================================================================
# 1. SETTINGS & CONSTANTS
# ===========================================================================

OUTDIR_BASE   = "squid_batch_analysis"
N_GRID        = 3000    # Points on the common interpolation grid
SAT_FRAC      = 0.05    # Outermost fraction of field used to fit background slope
STD_SMOOTH    = 61      # Moving-average window for SD band smoothing (odd)
DENSITY_G_CM3 = 15.0    # FePt bulk density [g/cm³]
HCP_PACKING   = 0.906   # Hexagonal close-packing fraction for sphere monolayer
OE_TO_T       = 1e-4    # Oersted → Tesla


# ===========================================================================
# 2. HELPERS
# ===========================================================================

def get_fept_volume_and_mass(substrate_area_mm2, sphere_diam_um, thick_nm):
    """
    Estimate total FePt volume and mass for an HCP monolayer of spheres.
    Returns: total_vol_cm3, mass_g, vol_per_cap_cm3, n_spheres
    """
    sphere_r_mm = (sphere_diam_um / 2) * 1e-3
    n_spheres   = (substrate_area_mm2 * HCP_PACKING) / (2 * np.sqrt(3) * sphere_r_mm**2)
    R_cm        = (sphere_diam_um / 2) * 1e-4
    t_cm        = thick_nm * 1e-7
    vol_cap_cm3 = (2/3) * np.pi * ((R_cm + t_cm)**3 - R_cm**3)
    total_vol   = n_spheres * vol_cap_cm3
    return total_vol, total_vol * DENSITY_G_CM3, vol_cap_cm3, n_spheres


def get_metrics(h, m, mass_g):
    """
    Extract coercivity [T], squareness (Mr/Ms), and hysteresis loss [J/kg].
    h in Oe, m in emu.
    """
    idx_hc      = np.argmin(np.abs(m))
    hc_t        = np.abs(h[idx_hc]) * OE_TO_T
    msat        = np.max(np.abs(m))
    mr          = np.interp(0, h[::-1], m[::-1])
    mr_ms       = np.abs(mr / msat)
    area_emu_oe = np.abs(np.trapezoid(m, h))
    whyst       = (area_emu_oe * 1e-7) / (mass_g * 1e-3)
    return hc_t, mr_ms, whyst


def split_branches(h, m):
    """
    Split a hysteresis loop into descending (H+→H−) and ascending (H−→H+)
    branches. Used only for computing the averaged mean ± SD — individual
    curves are plotted as a single continuous line.
    """
    idx_max, idx_min = np.argmax(h), np.argmin(h)
    if idx_max < idx_min:
        h_desc, m_desc = h[idx_max:idx_min+1], m[idx_max:idx_min+1]
        h_asc  = np.concatenate([h[idx_min:], h[:idx_max]])
        m_asc  = np.concatenate([m[idx_min:], m[:idx_max]])
    else:
        h_asc,  m_asc  = h[idx_min:idx_max+1], m[idx_min:idx_max+1]
        h_desc = np.concatenate([h[idx_max:], h[:idx_min]])
        m_desc = np.concatenate([m[idx_max:], m[:idx_min]])
    return (h_desc, m_desc), (h_asc, m_asc)


def moving_average(y, n):
    """Box-car moving average of window n (forced odd)."""
    if n < 3:
        return y
    if n % 2 == 0:
        n += 1
    return np.convolve(np.asarray(y), np.ones(n) / n, mode="same")


def subtract_linear_background(h, m):
    """
    Remove linear diamagnetic/paramagnetic substrate background.

    Fits the positive tail (h > 95% of h_max) and negative tail
    (h < −95% of h_max) independently, then averages the two slopes
    for robustness before subtracting.
    """
    h, m      = np.asarray(h), np.asarray(m)
    h_abs_max = np.max(np.abs(h))
    mask_pos  = h >  h_abs_max * (1 - SAT_FRAC)
    mask_neg  = h < -h_abs_max * (1 - SAT_FRAC)
    slope_pos, _ = np.polyfit(h[mask_pos], m[mask_pos], 1) if mask_pos.sum() > 2 else (0, 0)
    slope_neg, _ = np.polyfit(h[mask_neg], m[mask_neg], 1) if mask_neg.sum() > 2 else (0, 0)
    slope = (slope_pos + slope_neg) / 2
    return m - slope * h, slope


def plot_panel(ax, h_list, m_list, paths, colors,
               d_avg, d_sd, a_avg, a_sd,
               ylabel, title, show_legend=False):
    """
    Plot individual curves (continuous line, no desc/asc split) and
    the mean ± 1 SD band computed from averaged branches.
    """
    sm = lambda x: moving_average(x, STD_SMOOTH)
    for i, (h_r, m_r) in enumerate(zip(h_list, m_list)):
        ax.plot(h_r, m_r, color=colors[i % len(colors)], lw=1.2, alpha=0.8,
                label=os.path.basename(paths[i]) if show_legend else None)
    ax.fill_between(h_r*0 + ax.get_xlim()[0],  # dummy — replaced below
                    d_avg - sm(d_sd), d_avg + sm(d_sd),
                    color="black", alpha=0)       # placeholder, real calls below
    # replot correctly on h_grid (passed via closure — see main)
    ax._pp_args = (d_avg, d_sd, a_avg, a_sd)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.axhline(0, color="black", lw=0.5)
    ax.axvline(0, color="black", lw=0.5)
    ax.grid(True, ls=":", alpha=0.6)
    if show_legend:
        ax.legend(loc="lower right", fontsize=7)


# ===========================================================================
# 3. MAIN
# ===========================================================================

def main():

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    diam  = simpledialog.askfloat("Input", "Sphere diameter (µm):",  initialvalue=3.0)
    thick = simpledialog.askfloat("Input", "Cap thickness (nm):",    initialvalue=60.0)
    area  = simpledialog.askfloat("Input", "Substrate area (mm²):",  initialvalue=25.0)

    if any(v is None for v in [diam, thick, area]):
        messagebox.showwarning("Warning", "All geometry inputs are required.")
        root.destroy()
        return

    paths = filedialog.askopenfilenames(
        title=f"Select SQUID .dat files for {diam} µm batch"
    )
    root.destroy()

    if len(paths) < 2:
        print("[!] Please select 2 or more files for averaging.")
        return

    do_corr_str = input("Apply background subtraction? (yes/no): ").strip().lower()
    do_corr     = do_corr_str in ("yes", "y")

    batch_folder = os.path.join(OUTDIR_BASE, f"Batch_{diam}um_{ts}")
    os.makedirs(batch_folder, exist_ok=True)

    vol_cm3, mass_g, vol_cap_cm3, n_spheres = get_fept_volume_and_mass(area, diam, thick)
    print(f"\nFePt mass: {mass_g:.3e} g | Volume: {vol_cm3:.3e} cm³ | N particles: {n_spheres:.3e}")

    h_grid = None

    # Per-file accumulators (original sweep order, for plotting)
    all_raw_h    = []   # h in Tesla
    all_raw_m    = []   # raw emu
    all_raw_norm = []   # corrected + normalised M/Msat
    all_raw_pp   = []   # per-particle emu

    # Branch accumulators (for mean ± SD)
    all_rd, all_ra = [], []   # raw emu desc/asc
    all_nd, all_na = [], []   # normalised desc/asc
    all_pd, all_pa = [], []   # per-particle desc/asc

    results_list = []

    for p in paths:
        try:
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                lines = f.readlines()
            start_idx = next(
                (i + 1 for i, l in enumerate(lines) if "[Data]" in l), -1
            )
            if start_idx == -1:
                print(f"[!] No [Data] block in {os.path.basename(p)} — skipping.")
                continue

            df = pd.read_csv(p, skiprows=start_idx).dropna(
                subset=["Magnetic Field (Oe)", "Moment (emu)"]
            )
            h   = df["Magnetic Field (Oe)"].values
            m   = df["Moment (emu)"].values
            h_T = h * OE_TO_T

            if h_grid is None:
                h_grid = np.linspace(h_T.min(), h_T.max(), N_GRID)

            # Panel 1: raw emu
            all_raw_h.append(h_T)
            all_raw_m.append(m)
            (hdr, mdr), (har, mar) = split_branches(h_T, m)
            all_rd.append(np.interp(h_grid, hdr[np.argsort(hdr)], mdr[np.argsort(hdr)]))
            all_ra.append(np.interp(h_grid, har[np.argsort(har)], mar[np.argsort(har)]))

            # Background correction
            if do_corr:
                m_corr, slope = subtract_linear_background(h, m)
                print(f"  [{os.path.basename(p)}] slope removed: {slope:.4e} emu/Oe")
            else:
                m_corr = m.copy()

            # Panel 2: corrected + normalised
            msat   = np.max(np.abs(m_corr))
            m_norm = m_corr / msat
            all_raw_norm.append(m_norm)
            (hdn, mdn), (han, man) = split_branches(h_T, m_norm)
            all_nd.append(np.interp(h_grid, hdn[np.argsort(hdn)], mdn[np.argsort(hdn)]))
            all_na.append(np.interp(h_grid, han[np.argsort(han)], man[np.argsort(han)]))

            # Panel 3: per-particle moment
            m_pp = m_corr / n_spheres
            all_raw_pp.append(m_pp)
            (hdp, mdp), (hap, map_) = split_branches(h_T, m_pp)
            all_pd.append(np.interp(h_grid, hdp[np.argsort(hdp)], mdp[np.argsort(hdp)]))
            all_pa.append(np.interp(h_grid, hap[np.argsort(hap)], map_[np.argsort(hap)]))

            hc_t, mr_ms, whyst = get_metrics(h, m_corr, mass_g)
            results_list.append({
                "name": os.path.basename(p),
                "Hc_T": hc_t, "Mr_over_Ms": mr_ms, "Whyst_J_per_kg": whyst
            })

        except Exception as e:
            print(f"[!] Error in {os.path.basename(p)}: {e}")

    if not all_nd:
        print("[!] No files were successfully processed.")
        return

    # Statistics
    def stats(arr): return np.mean(arr, axis=0), np.std(arr, axis=0)
    rd_avg, rd_std = stats(all_rd);  ra_avg, ra_std = stats(all_ra)
    nd_avg, nd_std = stats(all_nd);  na_avg, na_std = stats(all_na)
    pd_avg, pd_std = stats(all_pd);  pa_avg, pa_std = stats(all_pa)
    sm = lambda x: moving_average(x, STD_SMOOTH)

    # ===========================================================================
    # 4. PLOTTING
    # ===========================================================================
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 15))
    fig.suptitle(
        f"FePt M(H) — {diam} µm batch ({len(paths)} samples)\n{ts}",
        fontweight="bold")

    def draw_panel(ax, h_list, m_list, d_avg, d_sd, a_avg, a_sd,
                   ylabel, title, show_legend=False):
        for i, (h_r, m_r) in enumerate(zip(h_list, m_list)):
            ax.plot(h_r, m_r, color=colors[i % len(colors)], lw=1.2, alpha=0.8,
                    label=os.path.basename(paths[i]) if show_legend else None)
        ax.fill_between(h_grid, d_avg - sm(d_sd), d_avg + sm(d_sd),
                         color="black", alpha=0.15)
        ax.fill_between(h_grid, a_avg - sm(a_sd), a_avg + sm(a_sd),
                         color="black", alpha=0.15, label=r"$\pm$1 SD")
        ax.plot(h_grid, d_avg, color="black", lw=2, label="Mean")
        ax.plot(h_grid, a_avg, color="black", lw=2)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.axhline(0, color="black", lw=0.5)
        ax.axvline(0, color="black", lw=0.5)
        ax.grid(True, ls=":", alpha=0.6)
        ax.legend(loc="lower right", fontsize=7)

    draw_panel(ax1, all_raw_h, all_raw_m,
               rd_avg, rd_std, ra_avg, ra_std,
               r"$m$ (emu)", "Raw hysteresis loop (no correction)",
               show_legend=True)

    draw_panel(ax2, all_raw_h, all_raw_norm,
               nd_avg, nd_std, na_avg, na_std,
               r"$M/M_{\mathrm{sat}}$",
               "Slope-corrected (diamagnetic Si substrate removed), normalized")

    draw_panel(ax3, all_raw_h, all_raw_pp,
               pd_avg, pd_std, pa_avg, pa_std,
               "Moment per particle (emu)",
               "Per-particle moment (HCP packing)")

    ax3.annotate(
        f"Sphere ⌀: {diam} µm   Cap: {thick} nm\n"
        f"HCP packing: {HCP_PACKING}   N particles: {n_spheres:.3e}",
        xy=(0.02, 0.05), xycoords="axes fraction", fontsize=8, color="gray",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
    ax3.set_xlabel(r"$\mu_0 H$ (T)")

    plt.tight_layout()
    base = os.path.join(batch_folder, f"Averaged_Batch_{diam}um_{ts}")
    fig.savefig(base + ".png", dpi=300)
    fig.savefig(base + ".svg")
    plt.close(fig)
    print(f"Plot saved.")

    # ===========================================================================
    # 5. REPORT & EXPORTS
    # ===========================================================================
    df_res = pd.DataFrame(results_list)
    with open(os.path.join(batch_folder, f"Report_{diam}um_{ts}.txt"), "w") as f:
        f.write("SQUID Analysis Report\n")
        f.write(f"Timestamp          : {ts}\n")
        f.write(f"Background corr    : {'Yes' if do_corr else 'No'}\n")
        f.write(f"HCP packing        : {HCP_PACKING}\n")
        f.write(f"N particles (est.) : {n_spheres:.4e}\n")
        f.write(f"FePt mass          : {mass_g:.6e} g\n")
        f.write(f"Vol per cap        : {vol_cap_cm3:.4e} cm³\n\n")
        f.write(df_res.to_string(index=False))
        f.write("\n\nRepeatability (mean ± std):\n")
        for col in ["Hc_T", "Mr_over_Ms", "Whyst_J_per_kg"]:
            avg, std = df_res[col].mean(), df_res[col].std()
            f.write(f"  {col:<20}: {avg:.6f} ± {std:.6f}\n")

    pd.DataFrame({
        "Field_T":                h_grid,
        "M_raw_desc_emu":         rd_avg,
        "M_raw_asc_emu":          ra_avg,
        "M_norm_desc":            nd_avg,
        "M_norm_asc":             na_avg,
        "M_perparticle_desc_emu": pd_avg,
        "M_perparticle_asc_emu":  pa_avg,
    }).to_csv(os.path.join(batch_folder, f"Averaged_Data_{diam}um_{ts}.csv"), index=False)

    print(f"\nDONE. All outputs saved in: {batch_folder}")


if __name__ == "__main__":
    main()
